# Submission 1 — Baseline LightGBM on Depth-0 Data

Uses only depth-0 tables (no joins to depth-1):
- `train_base` — case_id, WEEK_NUM, target
- `train_static_0_*` — 167 applicant-level features
- `train_static_cb_0` — 52 credit bureau static features

Key design choices:
- **Time-based CV** on WEEK_NUM (weeks 0-60 train, 61-91 validate) — no random splits
- **scale_pos_weight** to handle class imbalance (~3% default rate)
- Evaluated on the official **gini stability metric** (not just AUC)

In [ ]:
import polars as pl
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from scipy.stats import linregress
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA = '/kaggle/input/home-credit-credit-risk-model-stability/parquet_files'
OUTPUT = '/kaggle/working'

## 1. Load & join depth-0 tables

In [ ]:
base = pl.read_parquet(f'{DATA}/train/train_base.parquet')
static0 = pl.scan_parquet(f'{DATA}/train/train_static_0_*.parquet').collect()
static_cb = pl.read_parquet(f'{DATA}/train/train_static_cb_0.parquet')

df = (
    base
    .join(static0, on='case_id', how='left')
    .join(static_cb, on='case_id', how='left')
)

print(f'Train shape: {df.shape}')
print(f'Target rate: {df["target"].mean():.3%}')
print(f'WEEK_NUM range: {df["WEEK_NUM"].min()} - {df["WEEK_NUM"].max()}')

## 2. Load test data

In [ ]:
test_base = pl.read_parquet(f'{DATA}/test/test_base.parquet')
test_static0 = pl.scan_parquet(f'{DATA}/test/test_static_0_*.parquet').collect()
test_static_cb = pl.read_parquet(f'{DATA}/test/test_static_cb_0.parquet')

test = (
    test_base
    .join(test_static0, on='case_id', how='left')
    .join(test_static_cb, on='case_id', how='left')
)

print(f'Test shape: {test.shape}')

## 3. Preprocessing

- Date columns (`_D` suffix): parsed to days-since-epoch integer
- String/categorical columns: encoded as integer codes
- No imputation needed — LightGBM handles NaN natively

In [ ]:
DROP_COLS = ['case_id', 'date_decision', 'MONTH', 'WEEK_NUM', 'target']
TEST_DROP = ['case_id', 'date_decision', 'MONTH', 'WEEK_NUM']

def preprocess(df: pl.DataFrame, drop_cols: list) -> pl.DataFrame:
    date_cols = [c for c in df.columns if c.endswith('_D')]
    for col in date_cols:
        if col not in drop_cols:
            df = df.with_columns(
                pl.col(col).str.to_date(strict=False).cast(pl.Int32).alias(col)
            )
    cat_cols = [c for c in df.columns if df[c].dtype == pl.Utf8 and c not in drop_cols]
    for col in cat_cols:
        df = df.with_columns(
            pl.col(col).cast(pl.Categorical).to_physical().alias(col)
        )
    return df.drop([c for c in drop_cols if c in df.columns])

week_num = df['WEEK_NUM'].to_numpy()
y = df['target'].to_numpy()

X_df = preprocess(df, DROP_COLS)
X_test_df = preprocess(test, TEST_DROP)

feature_cols = [c for c in X_df.columns if c in X_test_df.columns]
X_df = X_df.select(feature_cols)
X_test_df = X_test_df.select(feature_cols)

X = X_df.to_numpy()
X_test = X_test_df.to_numpy()

print(f'Feature matrix: {X.shape}')
print(f'Test matrix:    {X_test.shape}')

## 4. Gini stability metric

Official competition metric: computes per-week Gini, fits a linear trend, and combines mean Gini + trend slope - residual variability.

In [ ]:
def gini_stability(week_nums, targets, preds):
    weeks = np.unique(week_nums)
    weekly_gini = []
    for w in weeks:
        mask = week_nums == w
        if targets[mask].sum() == 0:
            continue
        gini = 2 * roc_auc_score(targets[mask], preds[mask]) - 1
        weekly_gini.append(gini)
    weekly_gini = np.array(weekly_gini)
    mean_gini = weekly_gini.mean()
    slope, intercept, *_ = linregress(np.arange(len(weekly_gini)), weekly_gini)
    residuals = weekly_gini - (intercept + slope * np.arange(len(weekly_gini)))
    score = mean_gini + 88.0 * min(0, slope) - 0.5 * residuals.std()
    return score, mean_gini, slope, residuals.std(), weekly_gini

## 5. Time-based train/validation split

Weeks 0-60 → train, weeks 61-91 → validation. This mimics how the leaderboard evaluates the model on future weeks.

In [ ]:
TRAIN_CUTOFF = 60

train_mask = week_num <= TRAIN_CUTOFF
val_mask = week_num > TRAIN_CUTOFF

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
week_val = week_num[val_mask]

print(f'Train: {X_train.shape[0]:,} rows ({y_train.mean():.3%} positive)')
print(f'Val:   {X_val.shape[0]:,} rows ({y_val.mean():.3%} positive)')

## 6. Train LightGBM

In [ ]:
params = {
    'objective': 'binary',
    'metric': 'auc',
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'min_child_samples': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'scale_pos_weight': (y_train == 0).sum() / (y_train == 1).sum(),
    'n_jobs': -1,
    'verbose': -1,
    'random_state': 42,
}

model = lgb.LGBMClassifier(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)

print(f'Best iteration: {model.best_iteration_}')

## 7. Evaluate on validation set

In [ ]:
val_preds = model.predict_proba(X_val)[:, 1]

auc = roc_auc_score(y_val, val_preds)
score, mean_gini, slope, resid_std, weekly_gini = gini_stability(week_val, y_val, val_preds)

print(f'Validation AUC:        {auc:.4f}')
print(f'Gini stability score:  {score:.4f}')
print(f'  Mean weekly Gini:    {mean_gini:.4f}')
print(f'  Trend slope:         {slope:.6f}')
print(f'  Residual std:        {resid_std:.4f}')

In [ ]:
val_weeks = np.unique(week_val)
valid_weeks = [w for w in val_weeks if y_val[week_val == w].sum() > 0]

plt.figure(figsize=(10, 4))
plt.plot(valid_weeks, weekly_gini, marker='o', linewidth=1.5, label='Weekly Gini')
plt.axhline(mean_gini, color='red', linestyle='--', label=f'Mean = {mean_gini:.3f}')
trend = slope * np.arange(len(weekly_gini)) + (mean_gini - slope * len(weekly_gini) / 2)
plt.plot(valid_weeks, trend, color='orange', linestyle='--', label=f'Trend (slope={slope:.4f})')
plt.xlabel('WEEK_NUM')
plt.ylabel('Gini')
plt.title('Per-week Gini on validation set (weeks 61-91)')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Feature importance

In [ ]:
importance = pl.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort('importance', descending=True)

print('Top 20 features:')
print(importance.head(20))

plt.figure(figsize=(10, 8))
top20 = importance.head(20)
plt.barh(top20['feature'].to_list()[::-1], top20['importance'].to_list()[::-1])
plt.xlabel('Importance (split count)')
plt.title('Top 20 LightGBM feature importances')
plt.tight_layout()
plt.show()

## 9. Retrain on full data & generate submission

Retrain using all training data with the best number of estimators found during CV.

In [ ]:
final_params = {**params, 'n_estimators': model.best_iteration_}
final_model = lgb.LGBMClassifier(**final_params)
final_model.fit(X, y)

test_preds = final_model.predict_proba(X_test)[:, 1]

submission = pl.DataFrame({
    'case_id': test_base['case_id'],
    'score': test_preds
})

submission.write_csv(f'{OUTPUT}/submission.csv')
print(f'Saved {len(submission):,} predictions')
print(submission.describe())